In [1]:
import os, sys, importlib
sys.path.append(os.path.abspath(".."))
import json
import pandas as pd

In [2]:
results_dir=r"../B_images\images\results"
results=[]

for f in [f for f in os.listdir(results_dir) if f.endswith('.json')]:
    print(f)
    filename=os.path.splitext(f)[0]
    city=filename.split('_')[1]
    time=filename.split('_')[2]
    path_json=os.path.join(results_dir, f)
    print(path_json)
    with open (path_json, 'r', encoding="utf-8") as f:
        result_df=pd.DataFrame(json.load(f))
        result_df['in_paris']=1 if city=="paris" else 0
        result_df['time']=time
    
    results.append(result_df)

face_london_2306.json
../B_images\images\results\face_london_2306.json
face_london_2312.json
../B_images\images\results\face_london_2312.json
face_london_2406.json
../B_images\images\results\face_london_2406.json
face_paris_2306.json
../B_images\images\results\face_paris_2306.json
face_paris_2312.json
../B_images\images\results\face_paris_2312.json
face_paris_2406.json
../B_images\images\results\face_paris_2406.json


In [3]:
results_df=pd.concat(results, axis=0)
print(results_df.shape)
print(results_df[['time','in_paris']].value_counts(dropna=False))

(158179, 18)
time  in_paris
2406  1           43074
      0           24885
2312  1           23789
      0           22783
2306  0           21996
      1           21652
Name: count, dtype: int64


C:\Users\yeliu\AppData\Local\Temp\ipykernel_8452\1086956278.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df=pd.concat(results, axis=0)


In [4]:
print(results_df.isna().value_counts())
results_to_merge=results_df[['host_id',"in_paris",'time',"has_face","nb_face","host_picture_type"]]
display(results_to_merge.head())

img_path  host_id  has_face  nb_face  face_area_ratio  avg_face_prob  bbox_list  clean_score  lifestyle_score  host_picture_type  age    age_class  gender  smile_score  is_smiling  dominant_emotion  in_paris  time 
False     False    False     False    False            False          False      False        False            False              True   True       True    False        False       True              False     False    122217
                                                                                                                                  False  False      False   False        False       False             False     False     35871
                                                                                                                                  True   False      True    False        False       True              False     False        91
Name: count, dtype: int64


,host_id,in_paris,time,has_face,nb_face,host_picture_type
0,11535647,0,2306,1,1,life_style
1,9409420,0,2306,1,1,life_style
2,222363410,0,2306,1,1,pro_style
3,79823074,0,2306,1,1,pro_style
4,400346818,0,2306,1,1,pro_style


## merge avec listing_tactic


In [5]:
path_csv=r"../data_all\listings_tactics-paris_london-2306_2312_2406-1.csv"
df=pd.read_csv(path_csv)
print(df.shape)

C:\Users\yeliu\AppData\Local\Temp\ipykernel_8452\879773027.py:2: DtypeWarning: Columns (6,68,115) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path_csv)


(266852, 121)


In [7]:
df['time']=df['time'].astype(str)

In [8]:
df_bio_vis=df.merge(results_to_merge, on=['host_id','in_paris',"time"], how='left')


In [9]:
df_bio_vis[["in_paris",'time',"has_face","nb_face","host_picture_type"]].value_counts(dropna=False)


in_paris  time  has_face  nb_face  host_picture_type
1         2406  1.0       1.0      pro_style            24236
                0.0       0.0      no_person            16989
0         2406  1.0       1.0      pro_style            16328
                0.0       0.0      no_person            15871
          2312  1.0       1.0      pro_style            14308
                                                        ...  
1         2306  1.0       10.0     life_style               1
                          11.0     life_style               1
          2406  1.0       10.0     life_style               1
                          11.0     life_style               1
                          13.0     life_style               1
Name: count, Length: 75, dtype: int64

In [10]:
df_bio_vis.to_csv(r"../data_all\listings_tactics_bio_vis-paris_london-2306_2312_2406-1.csv", index=False)